# MMVEC Analysis of 16S V1-V3 and V4 and metabolomics table with top VIPs

Date created: 1/29/2024

Notebook author: Yang Chen

Data analysis by: Britta de Pessemier, Yang Chen

In [1]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
import qiime2
import os

In [2]:
_heatmap_choices = {
    'metric': {'braycurtis', 'canberra', 'chebyshev', 'cityblock',
               'correlation', 'cosine', 'dice', 'euclidean', 'hamming',
               'jaccard', 'kulsinski', 'mahalanobis', 'matching', 'minkowski',
               'rogerstanimoto', 'russellrao', 'seuclidean', 'sokalmichener',
               'sokalsneath', 'sqeuclidean', 'yule'},
    'method': {'single', 'complete', 'average', 'weighted', 'centroid',
               'median', 'ward'}}

_cmaps = {
    'heatmap': [
        'PiYG', 'PRGn', 'BrBG', 'PuOr', 'RdGy', 'RdBu',
        'RdYlBu', 'RdYlGn', 'Spectral', 'coolwarm', 'bwr', 'seismic',
        'viridis', 'plasma', 'inferno', 'magma', 'cividis'],
    'margins': [
        'cubehelix', 'Pastel1', 'Pastel2', 'Paired',
        'Accent', 'Dark2', 'Set1', 'Set2', 'Set3',
        'tab10', 'tab20', 'tab20b', 'tab20c',
        'viridis', 'plasma', 'inferno', 'magma', 'cividis']}

In [3]:
def _parse_heatmap_metadata_annotations(metadata_column, margin_palette):
    '''
    Transform feature or sample metadata into color vector for annotating
    margin of clustermap.
    Parameters
    ----------
    metadata_column: pd.Series of metadata for annotating plots
    margin_palette: str
        Name of color palette to use for annotating metadata
        along margin(s) of clustermap.
    Returns
    -------
    Returns vector of colors for annotating clustermap and dict mapping colors
    to classes.
    '''
    # Create a categorical palette to identify md col
    metadata_column = metadata_column.astype(str)
    col_names = sorted(metadata_column.unique())

    # Select Color palette
    if margin_palette == 'colorhelix':
        col_palette = sns.cubehelix_palette(
            len(col_names), start=2, rot=3, dark=0.3, light=0.8, reverse=True)
    else:
        col_palette = sns.color_palette(margin_palette, len(col_names))
    class_colors = dict(zip(col_names, col_palette))

    # Convert the palette to vectors that will be drawn on the matrix margin
    col_colors = metadata_column.map(class_colors)

    return col_colors, class_colors


def _parse_taxonomy_strings(taxonomy_series, level):
    '''
    taxonomy_series: pd.Series of semicolon-delimited taxonomy strings
    level: int
        taxonomic level for annotating clustermap.
     Returns
     -------
    Returns a pd.Series of taxonomy names at specified level,
        or terminal annotation
    '''
    return taxonomy_series.apply(lambda x: x.split(';')[:level][-1].strip())


def _process_microbe_metadata(ranks, microbe_metadata, level, margin_palette):
    _warn_metadata_filtering('microbe')
    microbe_metadata, ranks = microbe_metadata.align(
        ranks, join='inner', axis=0)
    # parse semicolon-delimited taxonomy
    if level > -1:
        microbe_metadata = _parse_taxonomy_strings(microbe_metadata, level)
    # map metadata categories to row colors
    row_colors, row_class_colors = _parse_heatmap_metadata_annotations(
        microbe_metadata, margin_palette)

    return microbe_metadata, ranks, row_colors, row_class_colors


def _process_metabolite_metadata(ranks, metabolite_metadata, margin_palette):
    _warn_metadata_filtering('metabolite')
    _ids = set(metabolite_metadata.index) & set(ranks.columns)
    ranks = ranks[sorted(_ids)]
    metabolite_metadata = metabolite_metadata.reindex(ranks.columns)
    # map metadata categories to column colors
    col_colors, col_class_colors = _parse_heatmap_metadata_annotations(
        metabolite_metadata, margin_palette)

    return metabolite_metadata, ranks, col_colors, col_class_colors


def _warn_metadata_filtering(metadata_type):
    warning = ('Conditional probabilities table and {0} metadata will be '
               'filtered to contain only the intersection of IDs in each. If '
               'this behavior is undesired, ensure that all {0} IDs are '
               'present in both the table and the metadata '
               'file'.format(metadata_type))
    warnings.warn(warning, UserWarning)


def _normalize_table(table, method):
    '''
    Normalize column data in a dataframe for plotting in clustermap.

    table: pd.DataFrame
        Input data.
    method: str
        Normalization method to use.

    Returns normalized table as pd.DataFrame
    '''
    if 'col' in method:
        axis = 0
    elif 'row' in method:
        axis = 1
    if 'z_score' in method:
        res = table.apply(lambda x: (x - x.mean()) / x.std(), axis=axis)
    elif 'rel' in method:
        res = table.apply(lambda x: x / x.sum(), axis=axis)
    elif method == 'log10':
        res = table.apply(lambda x: np.log10(x + 1))
    return res.fillna(0)

In [4]:
def ranks_heatmap(qza_path, microbe_metadata=None, metabolite_metadata=None,
                  method='average', metric='euclidean',
                  color_palette='seismic', margin_palette='cubehelix',
                  x_labels=False, y_labels=False, level=3, row_order=None, col_order=None, group=''):
    '''
    Generate clustermap of microbe X metabolite conditional probabilities.
    '''
    
    # Load the QIIME2 artifact and convert to Pandas DataFrame
    artifact = qiime2.Artifact.load(qza_path)  
    ranks = artifact.view(pd.DataFrame)  # Convert to Pandas DataFrame

    # Subset microbe metadata based on rows/columns
    if microbe_metadata is not None:
        microbe_metadata, ranks, row_colors, row_class_colors = _process_microbe_metadata(
            ranks, microbe_metadata, level, margin_palette)
    else:
        row_colors = None

    # Subset metabolite metadata based on rows/columns
    if metabolite_metadata is not None:
        metabolite_metadata, ranks, col_colors, col_class_colors = _process_metabolite_metadata(
            ranks, metabolite_metadata, margin_palette)
    else:
        col_colors = None

    # **Apply custom row & column ordering safely**
    if row_order is not None:
        row_order = [r for r in row_order if r in ranks.index]
        ranks = ranks.loc[row_order]

    if col_order is not None:
        col_order = [c for c in col_order if c in ranks.columns]
        ranks = ranks[col_order]

    # **Generate heatmap with smaller cells**
    hotmap = sns.clustermap(ranks, cmap=color_palette,
                            col_colors=col_colors, row_colors=row_colors,
                            figsize=(6, 6),  # Reduce figure size
                            method=method, metric=metric,
                            dendrogram_ratio=(0.03, 0.03),  # Reduce dendrogram size
                            row_cluster=False, col_cluster=False,  # Disable clustering to enforce order
                            cbar_kws={'label': 'Log Conditional Probability',
                                      'orientation': 'horizontal',
                                      'shrink': 0.5})

    # **Format Colorbar**
    hotmap.cax.set_position([0.25, -0.1, 0.5, 0.02])  # Move colorbar down
    cbar = hotmap.ax_heatmap.collections[0].colorbar

    cbar.set_label("Log Conditional Probability", fontsize=10, labelpad=10)
    cbar.ax.tick_params(labelsize=10)  # Increase tick font size

    # **Increase Axis Label Sizes**
    hotmap.ax_heatmap.tick_params(axis='x', labelsize=12)
    hotmap.ax_heatmap.tick_params(axis='y', labelsize=12)

    # **Move Title Higher and Update Group Name**
    if group == 'Acne_L':
        plt.suptitle('Acne Lesional', fontsize=20, y=1.08)
    elif group == 'Acne_NL':
        plt.suptitle('Acne Non-Lesional', fontsize=20, y=1.08)
    elif group == 'Healthy':
        plt.suptitle('Healthy', fontsize=20, y=1.08)        
    

    # **Move Row Legend to Lower Left**
    if row_colors is not None:
        hotmap.ax_row_dendrogram.legend(
            title=microbe_metadata.name, fontsize=10, loc="lower left"
        )

    # **Move Column Legend to Lower Left**
    if col_colors is not None:
        hotmap.ax_col_dendrogram.legend(
            title=metabolite_metadata.name, fontsize=10, loc="lower left"
        )

    # **Remove Y-axis Label**
    hotmap.ax_heatmap.set_ylabel("")

    # **Save the figure**
    plt.savefig(f'figures/mmvec_co-occurence_heatmap_{group}.png', dpi=600, bbox_inches="tight")
    plt.savefig(f'figures/mmvec_co-occurence_heatmap_{group}.svg')

    return hotmap


In [5]:
groups = ['Acne_L', 'Acne_NL', 'Healthy']

for skin_group in groups:
    print(f"Processing group: {skin_group}...")  # Debugging message

    color_palette = 'RdBu_r'

    col_order = [' g__Cutibacterium', ' g__Staphylococcus', ' g__Corynebacterium',
                 ' g__Streptococcus', ' g__Lawsonella', ' g__Micrococcus',
                 ' g__Haemophilus', ' g__Kocuria', ' g__Lactobacillus', ' g__Rothia',
                 ' g__Acinetobacter', ' g__Alloprevotella']
    
    row_order = ['Phenylalanine', 'D-TRYPTOPHAN', 'C19H36O4 Fatty acid', 'Nicotine',
                 'C19H38O4 Fatty acid', 'N-Oleoylethanolamine', 'Urocanic acid',
                 'Glutamyltyrosine', 'Sorbitane Monooleate', 'Gln-C14:0']
    
    # ranks_path = f'../Data/multi-omics/MMVEC/outputs/{skin_group}/exported_conditionals/conditionals.tsv'
    qza_path = f'outputs/{skin_group}/conditionals.qza'
    
    # ✅ **Check if file exists before proceeding**
    if not os.path.exists(qza_path):
        print(f"⚠️ Warning: File not found: {qza_path}. Skipping {skin_group}.")
        continue  # Skip this iteration if the file is missing
    

    # ✅ **Generate heatmap with filtered row/col order**
    ranks_heatmap(
        qza_path=qza_path,
        color_palette=color_palette,
        x_labels=True,
        y_labels=True,
        row_order=row_order,
        col_order=col_order,
        group=skin_group
    )

print("✅ All groups processed successfully.")


Processing group: Acne_L...
⚠️ Warning: File not found: outputs/Acne_L/conditionals.qza. Skipping Acne_L.
Processing group: Acne_NL...
⚠️ Warning: File not found: outputs/Acne_NL/conditionals.qza. Skipping Acne_NL.
Processing group: Healthy...
⚠️ Warning: File not found: outputs/Healthy/conditionals.qza. Skipping Healthy.
✅ All groups processed successfully.
